In [5]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install pandas
%pip install matplotlib
%pip install numpy



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [60]:
#creating a door to postgres with psycopg2, a function so that other functions can use the door without continiously needing to connect again.
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def connect_pgadmin():
    con= psycopg2.connect(
        host='localhost',
        database='DBM 0',
        user='postgres',
        password='Coderen73!',
        port='5432'
    )
    return con


conn = connect_pgadmin()
print("got it")
conn.close()


Matplotlib is building the font cache; this may take a moment.


got it


**1st Function: Call studio name and their respective average budget and average boxoffice (Stephan Muhin)**

This function was set up to answer parts of the following subquestion: 'To what extent are production-related factors associated with worldwide box-office performance?'. With this function you can for example get the data to furter analyse if a bigger budget means a bigger box_office on averag. 

In [44]:
def get_studio_performance ():
    conn= connect_pgadmin()
    cur= conn.cursor()
#creating a function, then opening the door to the database, and creating a cursor to scan through the data base 

    cur.execute('''
    select s.studio_name,
        count(*)as amount_movies,
        avg(b.production_budget) as av_budget,
        avg(b.worldwide_box_office) as av_boxoffice
    from movie m
    join box_office b on m.url = b.movie_url
    join studio s on m.studio_id = s.studio_id
    where b.production_budget is not null
    and b.worldwide_box_office is not null
    group by s.studio_name
    order by av_boxoffice desc
''')
# sql query to database, in which ask for the studio name
# count how many movvies there is, calc the average budget per studio, calc average box office per studio
# after joining 2 tables tother by url and same with studio_id, only selecting rows where budget and box office is nog null
# after group all films per studio, and sorting the from highest to lowest box office

    rows = cur.fetchall()
# takes all results and put them into the rows through the cursor
    columns = [
    'studio',
    'amount_movies',
    'av_budget',
    'av_boxoffice',
    ]
    df= pd.DataFrame(rows, columns=columns)
#puts all raw data in rows in the pandas table, and gives the columns names to create a clean table
    cur.close()
    conn.close()
#closes the connection with sql and leaves the cursor
    return df

In [49]:
df= get_studio_performance ()
print (df.head(10))
# defines the data frame and then prints the 10 highest scoring studio on boxoffice 

                                   studio  amount_movies  \
0     Walt Disney Studios Motion Pictures            111   
1          Village Roadshow Entertainment              1   
2              Warner Bros. Entertainment              1   
3                    Walt Disney Pictures              4   
4                            Warner Bros.             85   
5                             Belga Films              1   
6  Twentieth Century Fox Film Corporation            210   
7                       Columbia Pictures            206   
8     American Broadcasting Company (ABC)              2   
9                      Universal Pictures            287   

                av_budget        av_boxoffice  
0      136332432.43243243  499316580.88288288  
1   85000000.000000000000  385000317.00000000  
2   79000000.000000000000  318146162.00000000  
3  104750000.000000000000  216078432.25000000  
4   69236470.588235294118  215518258.00000000  
5  180000000.000000000000  215098356.00000000  
6  

In [ ]:
df= get_studio_performance ()
df['av_budget']= df['av_budget'].round(0)
df['av_boxoffice']= df['av_boxoffice'].round(0)
print (df.head(10))
# to round of the data

                                   studio  amount_movies  av_budget  \
0     Walt Disney Studios Motion Pictures            111  136332432   
1          Village Roadshow Entertainment              1   85000000   
2              Warner Bros. Entertainment              1   79000000   
3                    Walt Disney Pictures              4  104750000   
4                            Warner Bros.             85   69236471   
5                             Belga Films              1  180000000   
6  Twentieth Century Fox Film Corporation            210   60986190   
7                       Columbia Pictures            206   66483495   
8     American Broadcasting Company (ABC)              2   52500000   
9                      Universal Pictures            287   56366084   

  av_boxoffice  
0    499316581  
1    385000317  
2    318146162  
3    216078432  
4    215518258  
5    215098356  
6    207316587  
7    205661738  
8    183735928  
9    180958210  


**Linking data to literature review Ravid (1999)**

However if we want to test the existing theory from the theoretical review, literature states that big budget seem to predict bigger revenue. However, research states that it doesn't lead to higher profitablity. With the following query we can analyse the budget ratio per studio, to test whether higher spending actually leads to proportionally higher returns.

In [59]:
df['budget_ratio'] = (df['av_boxoffice']/ df['av_budget']).round(2)
print (df.head(10))

                                   studio  amount_movies  av_budget  \
0     Walt Disney Studios Motion Pictures            111  136332432   
1          Village Roadshow Entertainment              1   85000000   
2              Warner Bros. Entertainment              1   79000000   
3                    Walt Disney Pictures              4  104750000   
4                            Warner Bros.             85   69236471   
5                             Belga Films              1  180000000   
6  Twentieth Century Fox Film Corporation            210   60986190   
7                       Columbia Pictures            206   66483495   
8     American Broadcasting Company (ABC)              2   52500000   
9                      Universal Pictures            287   56366084   

  av_boxoffice budget_ratio  
0    499316581         3.66  
1    385000317         4.53  
2    318146162         4.03  
3    216078432         2.06  
4    215518258         3.11  
5    215098356         1.19  
6    207

**2nd Function: Call the director name and gives their respective boxoffice (Stephan Muhin)**

This function was also set up to answer a part of the following subquestion: 'To what extent are production-related factors associated with worldwide box-office performance?'. Now just like Zhao et al. (2016) in the literature review you can see if directors with a higher average gross cosistently generate more box office or not. 

In [62]:
def get_director_performance():
    conn=connect_pgadmin()
    cur=conn.cursor()
# created a function, then connected postgress and created a cursor so don't need to constantly open the door

    cur.execute('''
        select d.director_name,
            count(*) as amount_movies,
            avg(b.worldwide_box_office) as av_boxoffice
        from movie m
        join box_office b on m.url= b.movie_url
        join director d on m.director_id = d.director_id
        where b.worldwide_box_office is not null
        and m.director_id is not null
        group by d.director_name
        order by av_boxoffice desc
    ''')
# started with executing a function by asking sql for the director name
# then count the total amount of movies and create average of boxoffice per director
# in the movie table we join 2 table together by the url and 2 table together by the director id
# Then only selecting rows where the boxoffice is not null and the director id is also not null 
# then lastly grouping the all de movies per director and ordering the boxoffice from heighest to lowest

    rows = cur.fetchall()
    columns= [
        'director_name', 
        'amount_movies',
        'av_boxoffice'
        ]
# grabs all result selected by the cursor and puts them in rows
# also gives columns names to create a clean table

    df= pd.DataFrame(rows, columns=columns)
    cur.close()
    conn.close()
# puts all the raw data into a pandas table and asigns the new columns
# also closes the cursor and then also closes the door to the connection
    return df

In [ ]:
df= get_director_performance()
print (df.head(10))
# defines the data frame and then prints the 10 highest scoring studio on boxoffice 

    director_name  amount_movies         av_boxoffice
0      Chris Buck              2  1355389532.50000000
1      Kyle Balda              1  1159631140.00000000
2   Anthony Russo              6  1141248170.33333333
3     Josh Cooley              1  1073080329.00000000
4     Lee Unkrich              1  1068879522.00000000
5    Eric Guillon              1  1032961987.00000000
6     Joss Whedon              3   983578876.66666667
7       Jon Watts              4   974059770.75000000
8   James Cameron              3   962469179.33333333
9  Mike Thurmeier              1   879765137.00000000


In [ ]:
df= get_director_performance()
df['av_boxoffice']= df['av_boxoffice'].round(0)
print (df.head(10))
# to round of the data

    director_name  amount_movies av_boxoffice
0      Chris Buck              2   1355389532
1      Kyle Balda              1   1159631140
2   Anthony Russo              6   1141248170
3     Josh Cooley              1   1073080329
4     Lee Unkrich              1   1068879522
5    Eric Guillon              1   1032961987
6     Joss Whedon              3    983578877
7       Jon Watts              4    974059771
8   James Cameron              3    962469179
9  Mike Thurmeier              1    879765137
